# 01 - Raw Data Exploration


This notebook explores the raw Nginx logs collected from the Fink API before any parsing, preprocessing, or feature engineering.

The objectives are to understand the dataset structure and available log sources, inspect the number, size, distribution of files, examine representative samples from each log category and assess initial data quality.

The dataset contains 14 daily files per log category, with dates from October 4 to 17, 2026. Event timestamps range from October 3, 2026 at 00:00 UTC to October 17, 2026 at 00:00 UTC. Raw logs are stored locally and are not versioned in Git.

## Imports et configuration

In [4]:
import sys
from pathlib import Path
import pandas as pd
import humanize 

project_root = Path.cwd().parent 
sys.path.append(str(project_root))

from src.fink_api_anomaly_detection.data.raw_logs import (
    ls_log_files,
    read_gz_sample,
    inspect_logs_quality
)
    

For now, we're exploring the only nginx log data we have

In [5]:
DATA_DIR = project_root / "data" / "nginx_history_logs"
print(f"Dataset directory: {DATA_DIR.relative_to(project_root)}")

Dataset directory: data/nginx_history_logs


## Dataset Overview

In [6]:
categories = ["access-logs", "error-logs", "geo-logs"]

files_by_category = {
    category: ls_log_files(DATA_DIR, category)
    for category in categories
}

inventory = []

for category, files in files_by_category.items():
    for file_path in files:
        inventory.append(
            {
                "category": category,
                "filename": file_path.name,
                "path": str(file_path.relative_to(project_root)),
                "compressed_size_bytes": file_path.stat().st_size,
                "compressed_size_human": humanize.naturalsize(file_path.stat().st_size) ,
            }
        )

inventory_df = pd.DataFrame(inventory)

display(inventory_df)

,category,filename,path,compressed_size_bytes,compressed_size_human
0,access-logs,access.log-20261004.gz,data/nginx_history_logs/access-logs/access.log...,661158,661.2 kB
1,access-logs,access.log-20261005.gz,data/nginx_history_logs/access-logs/access.log...,1016256,1.0 MB
2,access-logs,access.log-20261006.gz,data/nginx_history_logs/access-logs/access.log...,558481,558.5 kB
3,access-logs,access.log-20261007.gz,data/nginx_history_logs/access-logs/access.log...,483589,483.6 kB
4,access-logs,access.log-20261008.gz,data/nginx_history_logs/access-logs/access.log...,487989,488.0 kB
5,access-logs,access.log-20261009.gz,data/nginx_history_logs/access-logs/access.log...,677648,677.6 kB
6,access-logs,access.log-20261010.gz,data/nginx_history_logs/access-logs/access.log...,1139352,1.1 MB
7,access-logs,access.log-20261011.gz,data/nginx_history_logs/access-logs/access.log...,516749,516.7 kB
8,access-logs,access.log-20261012.gz,data/nginx_history_logs/access-logs/access.log...,476911,476.9 kB
9,access-logs,access.log-20261013.gz,data/nginx_history_logs/access-logs/access.log...,294113,294.1 kB


In [7]:
print(f"Total compressed files: {len(inventory_df)}")

Total compressed files: 42


The dataset contains **42 compressed log files** collected over 14 days, with 14 files per category: access-logs, error-logs, and geo-logs.

### Log files Size Distribution

In [8]:
print(f"Total dataset files size : {humanize.naturalsize(inventory_df.compressed_size_bytes.sum())}")

for category, size in inventory_df.groupby("category")["compressed_size_bytes"].sum().items():
    print(f"Total {category} files size: {humanize.naturalsize(size)}")

Total dataset files size : 11.7 MB
Total access-logs files size: 8.4 MB
Total error-logs files size: 21.0 kB
Total geo-logs files size: 3.3 MB


In [9]:
inventory_df.groupby("category")["compressed_size_bytes"].describe()

,count,mean,std,min,25%,50%,75%,max
category,,,,,,,,
access-logs,14.0,600650.357143,243534.905136,294113.0,478580.50,537615.0,673525.50,1139352.0
error-logs,14.0,1501.428571,523.394206,704.0,1120.00,1343.0,1961.50,2370.0
geo-logs,14.0,236632.928571,117367.459278,133660.0,168499.25,192238.5,256027.75,558012.0


- **Access logs:** 14 files, averaging 600,650 bytes, with sizes ranging from 294,113 to 1,139,352 bytes.
- **Geo logs:** 14 files, averaging 236,633 bytes, with sizes ranging from 133,660 to 558,012 bytes.
- **Error logs:** 14 files, averaging 1,501 bytes, with sizes ranging from 704 to 2,370 bytes.

Access and geo logs show noticeable size variation across days, reflecting changes in request volume and activity.

# Raw Log Inspection

We now inspect representative samples from each log category. The purpose is to identify the general structure of each record and differences between log categories.

### sample file content from access category 

In [ ]:
sample_file = files_by_category["access-logs"][0]
print(f"First 10 lines from file: {sample_file.name} :\n")

first_10_lines = read_gz_sample(sample_file, n_lines=10)

for line in first_10_lines:
    print(line)

### sample file content from geo-logs category 

In [ ]:
sample_file = files_by_category["geo-logs"][0]
print(f"First 10 lines from file: {sample_file.name} :\n ")

first_10_lines = read_gz_sample(sample_file, n_lines=10)

for line in first_10_lines:
    print(line)

**Observation :** Access and geo logs can be successfully joined using the request timestamp, +xxxx, or request path. 

### sample file content from errors category 

In [ ]:
sample_file = files_by_category["error-logs"][0]
print(f"First 10 lines from file: {sample_file.name}\n")

first_10_lines = read_gz_sample(sample_file, n_lines=10)

for line in first_10_lines:
    print(line)

**Observation:** Error logs contain timestamps, client IPs, and requested endpoints, allowing them to be correlated with access logs. However, unlike access and geo logs, they do not share an identical format, so matching requires parsing and aligning these fields.

### Note

Log filenames are dated one day after the events they contain. For example, `access.log-20261004.gz` contains events starting on October 3 at 00:00 UTC. In the current dataset, the event timestamps range from October 3 at 00:00:59 UTC to October 17 at 00:00:14 UTC; the final date contains only six records.

## Cross-Day Log Format Consistency

In [ ]:
for category, files in files_by_category.items():
    print("-" * 80)
    print(f"CATEGORY: {category}")

    # Inspect the first and last available files
    selected_files = files[:1] + files[-1:] if len(files) > 1 else files

    for file_path in selected_files:
        print(f"\nFile: {file_path.name}")

        for line in read_gz_sample(file_path, n_lines=1):
            print(f" {line}")


Comparing files from different dates confirms that each log category follows a consistent structure across the dataset. Access, error, and geo logs retain their respective formats, with timestamps and request information aligned consistently across days.

## Raw Log Data Quality Assessment

In [14]:
quality_df = pd.DataFrame(inspect_logs_quality(files_by_category))

display(quality_df)

,category,filename,total_lines,empty_lines,min_line_length,max_line_length,mean_line_length
0,access-logs,access.log-20261004.gz,81778,0,83,595,137.601189
1,access-logs,access.log-20261005.gz,85204,0,82,595,151.735411
2,access-logs,access.log-20261006.gz,67708,0,81,594,141.142701
3,access-logs,access.log-20261007.gz,57646,0,83,576,143.134060
4,access-logs,access.log-20261008.gz,58906,0,69,575,142.214189
5,access-logs,access.log-20261009.gz,98556,0,83,678,151.155871
6,access-logs,access.log-20261010.gz,151645,0,83,575,162.983343
7,access-logs,access.log-20261011.gz,58005,0,83,595,144.644634
8,access-logs,access.log-20261012.gz,54177,0,84,1251,146.693486
9,access-logs,access.log-20261013.gz,31632,0,82,602,134.881639


All **42 log files** contain no empty lines. Across the 14 filename dates, access and geo logs have matching line counts, with **1,005,103 parsed records each**. The error logs contain **833 parsed records**. No unparseable lines were found in any category.